In [ ]:
import numpy as np
import optuna
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.panteeva.lesson3 import Exercise

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
n_ep = 50
rng = np.random.default_rng()


def train(trial):
    hidden_size = trial.suggest_int("hidden_layer_size", 1, 200)

    model = Exercise.create_model(
        Exercise.create_linear_layer(X_train.shape[1], hidden_size, rng=rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_size, len(np.unique(y_train)), rng=rng),
    )
    loss = Exercise.create_cross_entropy_loss()

    lr = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_int("batch_size", 1, 64)

    Exercise.train_model(
        model=model,
        loss=loss,
        x=X_train,
        y=y_train,
        lr=lr,
        n_epoch=n_ep,
        batch_size=batch_size,
    )
    val_predictions = model.forward(X_val)
    val_predicted_classes = np.argmax(val_predictions, axis=1)
    accuracy = np.mean(val_predicted_classes == y_val)

    return accuracy


study = optuna.create_study(direction="maximize")
study.optimize(train, n_trials=100)

study.best_value, study.best_params

In [ ]:
best_hs = study.best_params["hidden_layer_size"]
best_lr = study.best_params["learning_rate"]
best_bs = study.best_params["batch_size"]

final_model = Exercise.create_model(
    Exercise.create_linear_layer(X_train.shape[1], best_hs, rng=rng),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(best_hs, len(np.unique(y_train)), rng=rng),
)
final_loss = Exercise.create_cross_entropy_loss()

Exercise.train_model(
    model=final_model,
    loss=final_loss,
    x=X_train,
    y=y_train,
    lr=best_lr,
    n_epoch=n_ep,
    batch_size=best_bs,
)

test_predictions = final_model.forward(X_test)
test_predicted_classes = np.argmax(test_predictions, axis=1)
accuracy = np.mean(test_predicted_classes == y_test)
print(f"Test accuracy: {accuracy}")